# Beyond Audio — Reproducible Final Experiments: Stage 4
## Album-artwork overlap and near-duplicate sensitivity

**Purpose.** Reconstruct the original album-index and image-URL overlap checks; audit the already-identified perceptual-hash candidate pairs using the **current experimental split**; and measure how the original **frozen predictions** change when flagged test recordings are excluded.

This notebook is based on the original Colab development audit (particularly its corrected mapping and sensitivity steps) and uses the verified Stage 3 prediction-level results. It **does not** retrain models, reselect parameters, reassign visual-typicality quintiles, or alter the original report. Outputs are written to a separate `reproducible_stage4_2026` directory.

**Scientific scope:** A perceptual-hash candidate is a *possible* duplicate/similar cover, not proof that the original images are identical. This sensitivity check is retrospective; excluding flagged test observations does not establish that pretrained extractors have no overlap with the test data. The upstream creation and completeness of the near-duplicate candidate file are **not** independently reproduced here.

**Suggested workflow:** Run Sections 1–3 first and review the audit. Then reconstruct train–test pairs in Section 4 and examine the sensitivity results in Sections 5–6.

## 1. Connect Drive and check required inputs

In [1]:
from pathlib import Path
import os
import sys
import numpy as np
import pandas as pd

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")

PROJECT = Path(os.environ.get("MSDI_PROJECT_DIR", "/content/drive/MyDrive/MSD-I_project"))
CNN_DIR = PROJECT / "finetuning_experiments" / "multimodal_artist_disjoint"
MM_DIR = PROJECT / "finetuning_experiments" / "oramas_representation_experiment"
STAGE3_DIR = PROJECT / "finetuning_experiments" / "reproducible_stage3_2026"
ORIGINAL_AUDIT = PROJECT / "finetuning_experiments" / "methodology_audit"
OUTPUT = PROJECT / "finetuning_experiments" / "reproducible_stage4_2026"

required_files = {
    "original MSD-I metadata": PROJECT / "audio_features" / "MSD-I_dataset.tsv",
    "clean album metadata": PROJECT / "clean_album_metadata.csv",
    "perceptual-hash records": PROJECT / "album_perceptual_hashes.csv",
    "near-duplicate candidates": PROJECT / "near_duplicate_candidates.csv",
    "verified Stage 3 predictions": STAGE3_DIR / "stage3_paired_test_predictions_and_typicality.csv",
}
for split in ("train", "val", "test"):
    required_files[f"{split} labels"] = CNN_DIR / f"labels_{split}.csv"

missing = [f"{name}: {path}" for name, path in required_files.items() if not path.is_file()]
if missing:
    raise FileNotFoundError("Stage 4 input(s) missing:\n" + "\n".join(missing))

print("Project:", PROJECT)
print("Original audit folder:", ORIGINAL_AUDIT)
print("Verified Stage 3 input:", STAGE3_DIR)
print("Separate Stage 4 output (not yet created):", OUTPUT)
print("Python:", sys.version.split()[0], "NumPy:", np.__version__, "pandas:", pd.__version__)
print("Required Stage 4 files: FOUND")

Mounted at /content/drive
Project: /content/drive/MyDrive/MSD-I_project
Original audit folder: /content/drive/MyDrive/MSD-I_project/finetuning_experiments/methodology_audit
Verified Stage 3 input: /content/drive/MyDrive/MSD-I_project/finetuning_experiments/reproducible_stage3_2026
Separate Stage 4 output (not yet created): /content/drive/MyDrive/MSD-I_project/finetuning_experiments/reproducible_stage4_2026
Python: 3.13.18 NumPy: 2.4.6 pandas: 2.2.3
Required Stage 4 files: FOUND


## 2. Recheck the experimental splits and exact metadata overlaps

Use **current** `labels_train.csv`, `labels_val.csv`, and `labels_test.csv` for the splits. The original metadata supplies album IDs and image URLs. Exact overlaps are a narrower check than perceptual similarity.

In [2]:
# Stable, unambiguous representation for integer album IDs across CSVs.
def album_key(series):
    numeric = pd.to_numeric(series, errors="raise")
    if numeric.notna().any() and (numeric.dropna() % 1 != 0).any():
        raise ValueError("Album identifiers are not integers; inspect source metadata")
    return numeric.astype("Int64").astype("string")

split_frames = []
expected_sizes = {"train": 8242, "val": 1836, "test": 1792}
for split, expected in expected_sizes.items():
    frame = pd.read_csv(CNN_DIR / f"labels_{split}.csv", dtype={"msd_track_id": str, "msd_artist_id": str})
    assert len(frame) == expected, (split, len(frame), expected)
    assert {"msd_track_id", "msd_artist_id", "genre"}.issubset(frame.columns)
    assert frame["msd_track_id"].is_unique
    frame = frame[["msd_track_id", "msd_artist_id", "genre"]].copy()
    frame["current_split"] = split
    split_frames.append(frame)
labels_all = pd.concat(split_frames, ignore_index=True)
assert len(labels_all) == 11870 and labels_all["msd_track_id"].is_unique
artists_by_split = {
    split: set(part["msd_artist_id"])
    for split, part in labels_all.groupby("current_split")
}
assert all(artists_by_split[a].isdisjoint(artists_by_split[b])
           for a, b in [("train", "val"), ("train", "test"), ("val", "test")])

original_metadata = pd.read_csv(required_files["original MSD-I metadata"], sep="\t", low_memory=False,
                                dtype={"msd_track_id": str})
assert {"msd_track_id", "album_index", "image_url"}.issubset(original_metadata.columns)
metadata_for_join = original_metadata[["msd_track_id", "album_index", "image_url"]].copy()
if metadata_for_join["msd_track_id"].duplicated().any():
    grouped = metadata_for_join.groupby("msd_track_id", dropna=False)
    conflicts = grouped[["album_index", "image_url"]].nunique(dropna=False).max(axis=1)
    assert not (conflicts > 1).any(), "Conflicting metadata for repeated track IDs"
    metadata_for_join = metadata_for_join.drop_duplicates("msd_track_id")

subset = labels_all.merge(metadata_for_join, on="msd_track_id", how="left",
                          validate="one_to_one", indicator=True)
assert (subset["_merge"] == "both").all(), "Experimental tracks missing original metadata"
subset = subset.drop(columns="_merge")
assert subset["album_index"].notna().all()
subset["album_key"] = album_key(subset["album_index"])
subset["image_url_clean"] = subset["image_url"].astype("string").str.strip()
subset.loc[subset["image_url_clean"].isin(["", "nan", "None"]), "image_url_clean"] = pd.NA

pairs_splits = [("train", "val"), ("train", "test"), ("val", "test")]
exact_overlap_rows = []
for a, b in pairs_splits:
    A = subset[subset["current_split"] == a]
    B = subset[subset["current_split"] == b]
    ids = set(A["album_key"].dropna()) & set(B["album_key"].dropna())
    urls = set(A["image_url_clean"].dropna()) & set(B["image_url_clean"].dropna())
    exact_overlap_rows.append({"split_a": a, "split_b": b,
                               "shared_album_indices": len(ids), "shared_image_urls": len(urls),
                               "affected_tracks_a": int(A["image_url_clean"].isin(urls).sum()),
                               "affected_tracks_b": int(B["image_url_clean"].isin(urls).sum())})
exact_overlap = pd.DataFrame(exact_overlap_rows)
print("Experimental tracks / unique artists:", len(subset), subset["msd_artist_id"].nunique())
print("Artist-disjoint train/validation/test: VERIFIED")
print("Metadata match: ALL experimental tracks found")
print("Missing experimental image URLs:", int(subset["image_url_clean"].isna().sum()))
print("\nExact album-index and URL overlaps across CURRENT splits:")
print(exact_overlap.to_string(index=False))
if (exact_overlap[["shared_album_indices", "shared_image_urls"]] > 0).any().any():
    print("WARNING: exact cross-split overlap detected — investigate before interpreting robustness.")

Experimental tracks / unique artists: 11870 6461
Artist-disjoint train/validation/test: VERIFIED
Metadata match: ALL experimental tracks found
Missing experimental image URLs: 0

Exact album-index and URL overlaps across CURRENT splits:
split_a split_b  shared_album_indices  shared_image_urls  affected_tracks_a  affected_tracks_b
  train     val                     0                  0                  0                  0
  train    test                     0                  0                  0                  0
    val    test                     0                  0                  0                  0


## 3. Inspect candidate records and perceptual hashes

This step verifies the **existing** candidate file and independently calculates Hamming distances for those candidates where both stored hashes are available. It does not search every possible pair of artworks or re-download images.

In [3]:
clean_metadata = pd.read_csv(required_files["clean album metadata"], dtype={"msd_track_id": str})
candidates = pd.read_csv(required_files["near-duplicate candidates"])
hashes = pd.read_csv(required_files["perceptual-hash records"])
assert {"msd_track_id", "album_index"}.issubset(clean_metadata.columns)
assert {"album_a", "album_b", "distance"}.issubset(candidates.columns)
assert {"album_index", "phash"}.issubset(hashes.columns)
assert clean_metadata["album_index"].notna().all()

clean_metadata["album_key"] = album_key(clean_metadata["album_index"])
candidates["album_key_a"] = album_key(candidates["album_a"])
candidates["album_key_b"] = album_key(candidates["album_b"])
hashes["album_key"] = album_key(hashes["album_index"])

assert clean_metadata["album_key"].is_unique, "Clean metadata album IDs are not unique"
assert clean_metadata["msd_track_id"].is_unique, "Clean metadata track IDs are not unique"
assert hashes["album_key"].is_unique, "Multiple hash entries per album"
assert candidates[["album_key_a", "album_key_b", "distance"]].notna().all().all()

# Check that candidate album IDs resolve to unique tracks in the clean metadata.
album_to_track = clean_metadata.set_index("album_key")["msd_track_id"]
resolved_a = candidates["album_key_a"].map(album_to_track)
resolved_b = candidates["album_key_b"].map(album_to_track)

# Verify the album identifiers agree in clean and original metadata where tracks overlap.
clean_compare = clean_metadata[["msd_track_id", "album_key"]].merge(
    metadata_for_join[["msd_track_id", "album_index"]], on="msd_track_id", how="inner",
    validate="one_to_one"
)
clean_compare["original_album_key"] = album_key(clean_compare["album_index"])
metadata_disagreements = int((clean_compare["album_key"] != clean_compare["original_album_key"]).sum())

hash_text = hashes["phash"].astype("string").str.strip().str.lower().str.removeprefix("0x")
hash_lookup = dict(zip(hashes["album_key"], hash_text))
def hamming_hex(a, b):
    if pd.isna(a) or pd.isna(b) or str(a).strip() in ("", "nan") or str(b).strip() in ("", "nan"):
        return np.nan
    try:
        return (int(str(a), 16) ^ int(str(b), 16)).bit_count()
    except ValueError:
        return np.nan

calculated_distances = [hamming_hex(hash_lookup.get(a), hash_lookup.get(b))
                        for a, b in zip(candidates["album_key_a"], candidates["album_key_b"])]
candidates["verified_hamming_distance"] = calculated_distances
candidate_distance = pd.to_numeric(candidates["distance"], errors="raise")
compare_mask = candidates["verified_hamming_distance"].notna()
distance_mismatches = int((candidates.loc[compare_mask, "verified_hamming_distance"].to_numpy()
                           != candidate_distance.loc[compare_mask].to_numpy()).sum())

print("Candidate pairs in original file:", len(candidates))
print("Pairs with both album-to-track mappings:", int((resolved_a.notna() & resolved_b.notna()).sum()))
print("Clean/original album-ID disagreements where matched:", metadata_disagreements)
print("Candidates with both hashes available and valid:", int(compare_mask.sum()))
print("Stored vs reconstructed candidate hash-distance mismatches:", distance_mismatches)
print("\nCandidate perceptual-hash distance distribution:")
print(candidates["distance"].value_counts(dropna=False).sort_index().to_string())
if metadata_disagreements or distance_mismatches:
    print("WARNING: metadata or candidate-distance mismatches require investigation before proceeding.")
assert metadata_disagreements == 0, "Clean album IDs differ from original metadata"
assert distance_mismatches == 0, "Candidate distance differs from stored pHash distance"
print("Candidate source and identifier checks: PASSED")

Candidate pairs in original file: 12
Pairs with both album-to-track mappings: 12
Clean/original album-ID disagreements where matched: 0
Candidates with both hashes available and valid: 12
Stored vs reconstructed candidate hash-distance mismatches: 0

Candidate perceptual-hash distance distribution:
distance
0    3
2    5
4    4
Candidate source and identifier checks: PASSED


## 4. Reconstruct cross-split candidate pairs using **current** experiment assignments

**Critical correction:** `split_a` / `split_b` in the candidate file might describe an earlier split and must not be used to decide which test tracks to exclude. The current assignments come only from `labels_train/val/test.csv`. This was a corrected issue in the original audit.

In [4]:
expanded = candidates.copy()
expanded["track_a"] = expanded["album_key_a"].map(album_to_track)
expanded["track_b"] = expanded["album_key_b"].map(album_to_track)
current_split_lookup = labels_all.set_index("msd_track_id")["current_split"]
expanded["current_split_a"] = expanded["track_a"].map(current_split_lookup)
expanded["current_split_b"] = expanded["track_b"].map(current_split_lookup)

pair_mask = (
    (expanded["current_split_a"].eq("train") & expanded["current_split_b"].eq("test")) |
    (expanded["current_split_a"].eq("test") & expanded["current_split_b"].eq("train"))
)
train_test_candidates = expanded.loc[pair_mask].copy()
flagged_test_tracks = set(train_test_candidates.loc[
    train_test_candidates["current_split_a"].eq("test"), "track_a"
]) | set(train_test_candidates.loc[
    train_test_candidates["current_split_b"].eq("test"), "track_b"
])
assert flagged_test_tracks <= set(labels_all.loc[labels_all["current_split"].eq("test"), "msd_track_id"])
assert all(pd.notna(t) for t in flagged_test_tracks)

print("All candidate pairs:", len(expanded))
print("Candidates mapping to two tracks:", int(expanded[["track_a", "track_b"]].notna().all(axis=1).sum()))
print("Candidates with both tracks in current experiment:",
      int(expanded[["current_split_a", "current_split_b"]].notna().all(axis=1).sum()))
print("\nCurrent TRAIN–TEST candidate pairs:", len(train_test_candidates))
print("Unique flagged TEST recordings:", len(flagged_test_tracks))
print("\nCurrent train-test candidates (IDs and distances):")
print(train_test_candidates[["album_a", "album_b", "distance", "track_a", "track_b",
                             "current_split_a", "current_split_b"]].to_string(index=False))

# These are reference findings, not hard-coded output; if they change, investigate.
print("\nReference from original report: 5 candidate train-test pairs / 4 unique test tracks")
if len(train_test_candidates) != 5 or len(flagged_test_tracks) != 4:
    print("WARNING: counts differ from the earlier research audit. Review candidate mappings before proceeding.")
    raise AssertionError("Unexpected candidate counts; stop and investigate")

All candidate pairs: 12
Candidates mapping to two tracks: 12
Candidates with both tracks in current experiment: 12

Current TRAIN–TEST candidate pairs: 5
Unique flagged TEST recordings: 4

Current train-test candidates (IDs and distances):
 album_a  album_b  distance            track_a            track_b current_split_a current_split_b
   10865     6569         0 TRWVLQF128F424AFE3 TRRICCN12903CB4245            test           train
   20951    11162         2 TRIFHHG128F14565C7 TRYMZKX128F422403C           train            test
    5912    11162         2 TRVZSMT128F1456603 TRYMZKX128F422403C           train            test
   21502     9498         2 TRVYKJZ128F92F9D61 TRTOPTM128F92F2816           train            test
     969     6569         4 TRAYHYA12903D15D6A TRRICCN12903CB4245            test           train

Reference from original report: 5 candidate train-test pairs / 4 unique test tracks


## 5. Verify original frozen predictions and compute sensitivity

Read the verified Stage 3 file; **never refit models or change quintiles after test inspection**. Removing flagged recordings here is an exploratory, post-hoc sensitivity analysis, not a new independent test.

In [5]:
paired = pd.read_csv(required_files["verified Stage 3 predictions"],
                     dtype={"msd_track_id": str, "msd_artist_id": str})
needed = ["msd_track_id", "msd_artist_id", "genre", "typicality_quintile",
          "cnn_audio_correct", "cnn_early_fusion_correct", "mm_audio_correct", "mm_fusion_correct"]
assert set(needed).issubset(paired.columns), set(needed) - set(paired.columns)
assert len(paired) == 1792 and paired["msd_track_id"].is_unique

# Prove this Stage 3 snapshot belongs to the current test set and still has the right labels.
expected_test = labels_all.loc[labels_all["current_split"].eq("test"),
                               ["msd_track_id", "msd_artist_id", "genre"]]
match = paired[["msd_track_id", "msd_artist_id", "genre"]].merge(
    expected_test, on="msd_track_id", validate="one_to_one", how="outer",
    suffixes=("_stage3", "_labels"), indicator=True
)
assert len(match) == 1792 and (match["_merge"] == "both").all()
for field in ["msd_artist_id", "genre"]:
    assert (match[f"{field}_stage3"] == match[f"{field}_labels"]).all(), field
assert paired["typicality_quintile"].notna().all()
assert set(paired["typicality_quintile"].astype(int)) == {1, 2, 3, 4, 5}

for field in needed[4:]:
    # CSV boolean columns can be restored as bool or as 0/1.
    values = paired[field]
    if pd.api.types.is_bool_dtype(values):
        pass
    elif pd.api.types.is_numeric_dtype(values) and set(values.dropna().unique()).issubset({0,1}):
        paired[field] = values.astype(bool)
    elif set(values.dropna().unique()).issubset({"True", "False"}):
        paired[field] = values.map({"True": True, "False": False})
    else:
        raise ValueError(f"Unexpected correctness values in {field}: {values.unique()[:6]}")
    assert paired[field].notna().all()

paired["flagged_train_test_cover"] = paired["msd_track_id"].isin(flagged_test_tracks)
assert int(paired["flagged_train_test_cover"].sum()) == len(flagged_test_tracks)

comparisons = [("CNN", "cnn_audio_correct", "cnn_early_fusion_correct"),
               ("MM", "mm_audio_correct", "mm_fusion_correct")]
rows = []
for rep, audio_col, fusion_col in comparisons:
    for group_name, group in [
        ("Full test set", paired),
        ("Flagged train-test covers", paired[paired["flagged_train_test_cover"]]),
        ("Excluding flagged covers", paired[~paired["flagged_train_test_cover"]]),
    ]:
        audio = float(group[audio_col].mean()) if len(group) else np.nan
        fusion = float(group[fusion_col].mean()) if len(group) else np.nan
        rows.append({"representation": rep, "group": group_name, "n": len(group),
                     "audio_accuracy_pct": 100 * audio, "fusion_accuracy_pct": 100 * fusion,
                     "fusion_gain_pp": 100 * (fusion - audio)})
overall_sensitivity = pd.DataFrame(rows)
print("Original-prediction sensitivity: FULL vs FLAGGED vs EXCLUDING flagged")
print(overall_sensitivity.to_string(index=False, float_format=lambda x: f"{x:.2f}"))
print("\nReference finding: CNN gain excluding flagged covers approximately +3.64 pp")
cnn_unflagged = overall_sensitivity.query("representation == 'CNN' and group == 'Excluding flagged covers'")["fusion_gain_pp"].iloc[0]
print("Reconstructed CNN gain excluding flagged:", f"{cnn_unflagged:+.2f} pp")

Original-prediction sensitivity: FULL vs FLAGGED vs EXCLUDING flagged
representation                     group    n  audio_accuracy_pct  fusion_accuracy_pct  fusion_gain_pp
           CNN             Full test set 1792               65.90                69.64            3.74
           CNN Flagged train-test covers    4               50.00               100.00           50.00
           CNN  Excluding flagged covers 1788               65.94                69.57            3.64
            MM             Full test set 1792               66.52                70.20            3.68
            MM Flagged train-test covers    4               50.00               100.00           50.00
            MM  Excluding flagged covers 1788               66.55                70.13            3.58

Reference finding: CNN gain excluding flagged covers approximately +3.64 pp
Reconstructed CNN gain excluding flagged: +3.64 pp


## 6. Check the Q1–Q5 pattern and compare with the prior audit

Use Stage 3 quintile assignments unchanged. The number of rows per quintile may decrease after excluding flagged tracks. The analysis is descriptive; no bootstrap intervals are recomputed here.

In [6]:
q_rows = []
for rep, audio_col, fusion_col in comparisons:
    for q in range(1, 6):
        full = paired[paired["typicality_quintile"].eq(q)]
        clean = full[~full["flagged_train_test_cover"]]
        def gain(group):
            return 100 * (group[fusion_col].mean() - group[audio_col].mean()) if len(group) else np.nan
        q_rows.append({"representation": rep, "quintile": q,
                       "n_full": len(full), "n_excluding_flagged": len(clean),
                       "gain_full_pp": gain(full), "gain_excluding_flagged_pp": gain(clean),
                       "removed_test_tracks": len(full) - len(clean)})
quintile_sensitivity = pd.DataFrame(q_rows)
print("Fusion gains by original genre-relative quintile (percentage points):")
print(quintile_sensitivity.to_string(index=False, float_format=lambda x: f"{x:.2f}"))

# Compare the original CNN-only audit if present; it used fractions, not percentage units.
original_csv = ORIGINAL_AUDIT / "cover_similarity_sensitivity_overall.csv"
if original_csv.is_file():
    old = pd.read_csv(original_csv)
    subset_cnn = overall_sensitivity.loc[overall_sensitivity["representation"].eq("CNN")]
    current = subset_cnn.set_index("group")
    old = old.set_index("group")
    compare_names = sorted(set(current.index) & set(old.index))
    if {"audio_accuracy", "fusion_accuracy", "fusion_minus_audio"}.issubset(old.columns):
        diffs = []
        for group in compare_names:
            diffs.extend([abs(current.loc[group, "audio_accuracy_pct"] - 100 * old.loc[group, "audio_accuracy"]),
                          abs(current.loc[group, "fusion_accuracy_pct"] - 100 * old.loc[group, "fusion_accuracy"]),
                          abs(current.loc[group, "fusion_gain_pp"] - 100 * old.loc[group, "fusion_minus_audio"])])
        max_diff = max(diffs) if diffs else np.nan
        print("\nComparison with original CNN sensitivity audit (max difference in pp):", max_diff)
        assert max_diff < 1e-8, "CNN sensitivity values differ from original audit"
    else:
        print("\nOriginal CNN audit present but columns differ; review manually:", old.columns.tolist())
else:
    print("\nOriginal CNN-only sensitivity CSV not found; independent results shown above.")

print("\nImportant: candidate threshold and hash generation were not reimplemented; this is a sensitivity check using saved candidate pairs.")

Fusion gains by original genre-relative quintile (percentage points):
representation  quintile  n_full  n_excluding_flagged  gain_full_pp  gain_excluding_flagged_pp  removed_test_tracks
           CNN         1     361                  358         17.45                      17.04                    3
           CNN         2     357                  357         12.32                      12.32                    0
           CNN         3     357                  356         10.64                      10.67                    1
           CNN         4     357                  357         -1.96                      -1.96                    0
           CNN         5     360                  360        -19.72                     -19.72                    0
            MM         1     361                  358         17.45                      17.04                    3
            MM         2     357                  357         12.89                      12.89                    0
  

## 7. Save new Stage 4 evidence without modifying the originals

In [7]:
OUTPUT.mkdir(parents=True, exist_ok=True)
exact_overlap.to_csv(OUTPUT / "stage4_exact_overlap.csv", index=False)
expanded.to_csv(OUTPUT / "stage4_candidate_mapping.csv", index=False)
train_test_candidates.to_csv(OUTPUT / "stage4_current_train_test_cover_candidates.csv", index=False)
paired[["msd_track_id", "genre", "msd_artist_id", "typicality_quintile", "flagged_train_test_cover"]].to_csv(
    OUTPUT / "stage4_flagged_test_tracks.csv", index=False
)
overall_sensitivity.to_csv(OUTPUT / "stage4_overall_sensitivity.csv", index=False)
quintile_sensitivity.to_csv(OUTPUT / "stage4_quintile_sensitivity.csv", index=False)

print("Saved six Stage 4 CSVs to:", OUTPUT)
print("No original datasets, predictions, model checkpoints, or previous audit files were overwritten.")
print("\nVerification summary")
print("  Test artists:", paired["msd_artist_id"].nunique())
print("  Candidate current train-test pairs:", len(train_test_candidates))
print("  Unique flagged test tracks:", len(flagged_test_tracks))
print("  CNN fusion gain excluding flagged:", f"{cnn_unflagged:+.2f} pp")
print("  Primary limitation: known candidate pairs only; no independent upstream hash generation audit.")

Saved six Stage 4 CSVs to: /content/drive/MyDrive/MSD-I_project/finetuning_experiments/reproducible_stage4_2026
No original datasets, predictions, model checkpoints, or previous audit files were overwritten.

Verification summary
  Test artists: 975
  Candidate current train-test pairs: 5
  Unique flagged test tracks: 4
  CNN fusion gain excluding flagged: +3.64 pp
  Primary limitation: known candidate pairs only; no independent upstream hash generation audit.


## Before updating your paper or GitHub verification report

Confirm that the outputs match the earlier analysis: **five** candidate train–test cover pairs, **four** unique flagged test tracks, and a CNN gain of about **+3.64 percentage points** after those four tracks are excluded. Check the Q1/Q5 figures, too.

**Do not** interpret exclusion as proof of no leakage or as an independently selected test set. In particular, this notebook cannot independently audit the original pretrained CNN/MM feature extractors or guarantee that perceptual-hash candidates capture all visually similar artwork. Any non-matching value should be investigated rather than manually adjusted.